# Missing BUSCOs in yeast
This is a follow up analysis on the saccharomycetes annotations that showed an increase in completness using the specific busco set. 

In [1]:
# build the command file for available saccaromicetes using the euka busco set and save them in saccharomycetes specific result folder
# the annotations are the Saccharomyces ones of the core set (core_annotations.tsv), run with the generic eukaryota lineage
# the run_busco_saccharomycetes.py output of each annotation is results_saccharomycetes/<annotation_id>_eukaryota_odb12/
from pathlib import Path

import pandas as pd

LINEAGES_DIR = Path("/no_backup/rg/references/busco_downloads/lineages")
LINEAGE = "eukaryota_odb12"
RESULTS_DIR = Path("results_saccharomycetes")
COMMANDS_FILE = Path("run_busco_commands_saccharomycetes.txt")
ACTIVATE_ENV = "source ~/miniconda3/etc/profile.d/conda.sh && conda activate busco-annotrieve"
CPU = 4

lineage_path = LINEAGES_DIR / LINEAGE
assert lineage_path.is_dir(), f"missing lineage {lineage_path}"
RESULTS_DIR.mkdir(exist_ok=True)

core = pd.read_csv("core_annotations.tsv", sep="\t")
annotation_ids = core.loc[core["group"] == "Saccharomyces", "annotation_id"]

with open(COMMANDS_FILE, "w") as f:
    for annotation_id in annotation_ids:
        f.write(f"{ACTIVATE_ENV} && python run_busco_saccharomycetes.py {annotation_id} {lineage_path} "
                f"--cpu {CPU} -o {RESULTS_DIR}/{annotation_id}\n")

print(f"{len(annotation_ids)} commands written to {COMMANDS_FILE}, results in {RESULTS_DIR}/")


8 commands written to run_busco_commands_saccharomycetes.txt, results in results_saccharomycetes/


In [2]:
# extract the missing busco genes from the result folder
import glob

import pandas as pd

rows = []
for path in glob.glob("results_saccharomycetes/*/busco_*/run_*/missing_busco_list.tsv"):
    run = path.split("/")[1]
    species, assembly = glob.glob(f"results_saccharomycetes/{run}/annotrieve/*/*")[0].split("/")[-2:]
    busco_ids = pd.read_csv(path, sep="\t", comment="#", names=["busco_id"])["busco_id"]
    rows += [(f"{species} {assembly}", busco_id) for busco_id in busco_ids]

missing = pd.DataFrame(rows, columns=["annotation", "busco_id"])
missing.groupby("annotation").size().rename("n_missing")

annotation
Saccharomyces_arboricola_H-6_1160507 GCA_000292725.1          31
Saccharomyces_cerevisiae_4932 GCA_949125895.1                  9
Saccharomyces_eubayanus_1080349 GCA_946408725.1                8
Saccharomyces_kudriavzevii_IFO_1802_226230 GCF_947243775.1     8
Saccharomyces_mikatae_IFO_1815_226126 GCF_947241705.1          9
Saccharomyces_paradoxus_27291 GCA_002079055.1                  9
Saccharomyces_pastorianus_27292 GCA_011022315.1                9
Saccharomyces_uvarum_230603 GCA_027557585.1                    9
Name: n_missing, dtype: int64

In [3]:
# identify the missing gene in common to all anotations
n_annotations = missing["annotation"].nunique()

# in how many annotations is each busco missing?
n_missing_in = missing.groupby("busco_id").size().rename("n_annotations_missing")

common_missing = n_missing_in[n_missing_in == n_annotations]
print(f"{len(common_missing)} BUSCOs are missing in all {n_annotations} annotations")
print(f"{len(n_missing_in) - len(common_missing)} are missing in only some of them")

8 BUSCOs are missing in all 8 annotations
24 are missing in only some of them


In [4]:
# check for the function of the functional description of these genes
# descriptions are in the lineage dataset (full_table.tsv leaves them empty for missing genes)
lineage_links = "/no_backup/rg/references/busco_downloads/lineages/eukaryota_odb12/links_to_ODB12.txt"
descriptions = pd.read_csv(lineage_links, sep="\t", names=["busco_id", "description", "orthodb_url"], index_col="busco_id")

pd.set_option("display.max_colwidth", 100)
n_missing_in.to_frame().join(descriptions).sort_values("n_annotations_missing", ascending=False)

,n_annotations_missing,description,orthodb_url
busco_id,,,
10261918at2759,8,splicing factor,https://v12-1.orthodb.org/?query=10261918at2759
1898821at2759,8,BETA-CATENIN-LIKE PROTEIN 1,https://v12-1.orthodb.org/?query=1898821at2759
15082at2759,8,Eukaryotic translation initiation factor 3 subunit,https://v12-1.orthodb.org/?query=15082at2759
163257at2759,8,SERINE/ARGININE REPETITIVE MATRIX PROTEIN 1,https://v12-1.orthodb.org/?query=163257at2759
190958at2759,8,Pre-mRNA-splicing factor,https://v12-1.orthodb.org/?query=190958at2759
272987at2759,8,Conserved oligomeric Golgi complex subunit 6,https://v12-1.orthodb.org/?query=272987at2759
25498at2759,8,eukaryotic translation initiation factor 3 subunit,https://v12-1.orthodb.org/?query=25498at2759
2083at2759,8,peptidyl-prolyl cis-trans isomerase,https://v12-1.orthodb.org/?query=2083at2759
10252281at2759,4,Asparagine synthetase domain-containing protein,https://v12-1.orthodb.org/?query=10252281at2759
